# Tree diagnostics: is the supermatrix any good?

One notebook for **all tree variants**, so they can be compared on the same page. It is executed once by the
`tree_diagnostics` rule after every alignment exists, and looks at the **concatenated alignments** (before any
tree is built). The tree side (fragmentation, support, RED) is in `tree_accuracy`.

Variants are labelled `derep/copies`:

| part | values | meaning |
|---|---|---|
| derep | `all` / `derep` | every genome, or only the dereplicated representatives (external genomes always kept) |
| copies | `single` / `multi` | Complete BUSCO genes only, or also the top-scoring copy of Duplicated genes |

A quick-check subset run (label printed below) uses far fewer genomes: do not compare its numbers with a full run.

Questions answered (each section is mostly figures; the numbers are also written as TSV next to the notebook):

1. **Occupancy** - does every genome have enough genes, or are some mostly gaps?
2. **Divergence and gene overlap** - are genomes of the same clade closer, and do they share more genes, than genomes of different clades?
3. **Per-gene separation** - does any gene fail to tell clades apart?
4. **Single vs multi copy** - do the extra Duplicated genes help?
5. **Multicopy rule** - is "take the top-scoring copy" a good choice?

## 0. Setup

Parameters are written by the `tree_diagnostics` rule into a JSON file (path in `NB_PARAMS`), so the notebook has
no hard-coded paths. The statistics live in `scripts/tree_diagnostics_lib.py` (unit-tested); this notebook loads the
data, calls them and draws the results.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
PARAMS = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, PARAMS["scripts_dir"])
import tree_diagnostics_lib as lib

OUT = Path(PARAMS["out_dir"])        # tables and PNG copies of the figures
OUT.mkdir(parents=True, exist_ok=True)
RANKS = ["phylum", "class", "order", "family", "genus"]
N_PAIRS = PARAMS["n_pairs"]          # genome pairs sampled per rank
MIN_PAIRS = 30                       # within-clade pairs with data needed to score a gene
VARIANTS = [v for v in PARAMS["variants"] if Path(v["alignment"]).exists()]
LABELS = [v["label"] for v in VARIANTS]
COLOURS = dict(zip(LABELS, plt.cm.tab10.colors))   # one colour per variant in every figure

print("Run     :", PARAMS["label"])
print("Variants:", ", ".join(LABELS))

In [ ]:
def show_all(df, width=200):
    '''
    Prints an entire polars dataframe in the notebook output.

    Parameters
    ----------
    df : pl.DataFrame
        The dataframe to be printed.
    width : int, optional
        The width (in characters) of the printed dataframe.
        Defaults to 200.
    '''
    with pl.Config() as cfg:
        cfg.set_tbl_cols(-1)
        cfg.set_tbl_rows(-1)
        cfg.set_tbl_width_chars(width)
        cfg.set_fmt_str_lengths(width)
        print(df)


def finish(fig, name):
    '''
    Saves a figure as PNG next to the tables and shows it in the notebook.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        The figure to save.
    name : str
        File name without extension.
    '''
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

## 1. Load every alignment

For each variant: the genome x column matrix (`-` = gap), the gene boundaries, and the sampled genome pairs
used in sections 2-4. Pairs are sampled within and between clades at every rank, with the same seed for every
variant, and scored with `p-distance` (differing residues over columns where both genomes have a residue),
`jaccard` (shared genes / genes in either genome) and `shared_genes`.

In [ ]:
taxonomy = lib.read_taxonomy(PARAMS["taxonomy"])   # {genome: {rank: name}} database lineages

DATA = {}   # label -> everything computed for that variant
for variant in VARIANTS:
    ids, matrix = lib.load_alignment(variant["alignment"])
    partitions = lib.read_partitions(variant["partitions"])
    occupancy, present = lib.occupancy(ids, matrix, partitions)
    report = lib.rank_report(ids, matrix, present, partitions, taxonomy,
                             ranks=RANKS, n_pairs=N_PAIRS, seed=1)
    gene_rows = [{"variant": variant["label"], "rank": rank, "gene": gene,
                  "separation": separation, "within_pairs_with_data": n}
                 for rank in report for gene, separation, n in lib.gene_separation(report, partitions, rank)]
    DATA[variant["label"]] = {
        "ids": ids, "partitions": partitions,
        "occupancy": pl.DataFrame(occupancy).with_columns(pl.lit(variant["label"]).alias("variant")),
        "genomes_per_gene": present.sum(axis=0),
        "report": {rank: {kind: {k: report[rank][kind][k] for k in ("pdist", "jaccard", "shared_genes")}
                          for kind in ("within", "between")} | {"summary": report[rank]["summary"]}
                   for rank in report},
        "gene_separation": pl.DataFrame(gene_rows),
    }
    print(f"{variant['label']:14} {len(ids):6d} genomes x {matrix.shape[1]} columns, {len(partitions)} genes")
    del matrix

## 2. Occupancy: genes per genome and gaps

Each genome's row of the supermatrix: how many genes it has at all, and what fraction of the row is gap.

**How to read it.** Most genomes should have well over half of the genes. A long tail with few genes adds mostly
gaps to the supermatrix; such genomes are placed unstably and can look "misplaced". The workflow already drops
genomes below `alignment.min_genes_per_genome`; if many rows are still above ~90% gaps, raise that threshold or
look at those genomes' BUSCO completeness. `genomes per gene` should be high for every gene; a gene present in
few genomes contributes little.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for label in LABELS:
    occ = DATA[label]["occupancy"]
    kw = dict(histtype="step", linewidth=1.6, color=COLOURS[label], label=label)
    axes[0].hist(occ["genes"], bins=40, **kw)
    axes[1].hist(occ["gap_fraction"], bins=40, **kw)
    axes[2].hist(DATA[label]["genomes_per_gene"], bins=40, **kw)
axes[0].set_title("genes per genome")
axes[1].set_title("gap fraction of each genome's row")
axes[2].set_title("genomes per gene")
axes[0].legend(fontsize=8)
finish(fig, "occupancy")

occupancy_all = pl.concat([DATA[label]["occupancy"] for label in LABELS])
occupancy_all.write_csv(OUT / "occupancy.tsv", separator="\t")

The genomes with the fewest genes are the first candidates to inspect. The bar length is the number of genes;
the colour is the variant. (Full list: `occupancy.tsv`.)

In [ ]:
N_SHOWN = 20
fewest = occupancy_all.sort("genes").group_by("genome", maintain_order=True).first().head(N_SHOWN)
fig, axis = plt.subplots(figsize=(8, 0.3 * N_SHOWN + 1.5))
phylum = lambda genome: taxonomy.get(genome, {}).get("phylum", "unlabelled")
axis.barh(range(N_SHOWN), fewest["genes"], color=[COLOURS[v] for v in fewest["variant"]])
axis.set_yticks(range(N_SHOWN))
axis.set_yticklabels([f"{g}  ({phylum(g)})" for g in fewest["genome"]], fontsize=8)
axis.invert_yaxis()
axis.set_xlabel("genes present")
axis.set_title(f"{N_SHOWN} genomes with the fewest genes")
finish(fig, "fewest_genes")

## 3. Do clades share genes, and are they closer in sequence?

For genome pairs at each rank, **within** = same clade name, **between** = different names. Lines are medians and
the bars the inter-quartile range.

* **p-distance** - within-clade distance should fall steadily from phylum to genus while between-clade distance
  stays high. Once between-clade distance stops growing with depth, p-distance is saturated and cannot order the
  deep divergences; the tree then leans on the substitution model.
* **separation** - probability that a within pair is closer than a between pair (1.0 = clades perfectly
  distinguishable, 0.5 = no information, below 0.5 = inverted). If it is high but the tree still fragments a clade,
  suspect the labels, a few misplaced genomes or the tree model rather than the data.
* **gene-set Jaccard / shared genes** - within above between means clade members have similar gene sets. That can be
  real, but it can also mean *missing* genes follow the clade, so genomes could group by what they lack. Read it
  with section 2.

In [ ]:
def rank_line(axis, key, title, ylabel):
    '''
    Draws the median (and inter-quartile range) of a pair statistic against rank for every variant.

    Parameters
    ----------
    axis : matplotlib.axes.Axes
        Axis to draw on.
    key : str
        Pair statistic: "pdist", "jaccard" or "shared_genes".
    title, ylabel : str
        Axis title and y label. Solid lines are within-clade pairs, dashed are between-clade pairs.
    '''
    x = np.arange(len(RANKS))
    for label in LABELS:
        for kind, style in (("within", "-"), ("between", "--")):
            q = np.array([np.nanquantile(np.asarray(DATA[label]["report"][r][kind][key], dtype=float), [0.25, 0.5, 0.75])
                          for r in RANKS if r in DATA[label]["report"]])
            xs = x[:len(q)]
            axis.errorbar(xs, q[:, 1], yerr=[q[:, 1] - q[:, 0], q[:, 2] - q[:, 1]], linestyle=style,
                          color=COLOURS[label], marker="o", markersize=3, capsize=2, linewidth=1.2,
                          label=f"{label} {kind}" if key == "pdist" else None)
    axis.set_xticks(x)
    axis.set_xticklabels(RANKS)
    axis.set_title(title)
    axis.set_ylabel(ylabel)


fig, axes = plt.subplots(2, 2, figsize=(13, 8))
rank_line(axes[0, 0], "pdist", "p-distance (shared sites)", "p-distance")
rank_line(axes[0, 1], "jaccard", "gene-set Jaccard", "shared / union of genes")
rank_line(axes[1, 0], "shared_genes", "genes shared by the pair", "genes")
for label in LABELS:
    sep = [DATA[label]["report"][r]["summary"]["separation_pdist"] for r in RANKS if r in DATA[label]["report"]]
    axes[1, 1].plot(range(len(sep)), sep, marker="o", color=COLOURS[label], label=label)
axes[1, 1].axhline(0.5, color="grey", linestyle=":")
axes[1, 1].set_xticks(range(len(RANKS)))
axes[1, 1].set_xticklabels(RANKS)
axes[1, 1].set_ylim(0.45, 1.01)
axes[1, 1].set_title("separation (within pair closer than between pair)")
axes[0, 0].legend(fontsize=7, ncol=2)
finish(fig, "rank_divergence")

rank_summary = pl.concat([pl.DataFrame([DATA[l]["report"][r]["summary"] for r in RANKS if r in DATA[l]["report"]])
                          .with_columns(pl.lit(l).alias("variant")) for l in LABELS])
rank_summary.write_csv(OUT / "rank_summary.tsv", separator="\t")

## 4. Per-gene separation

The same within/between comparison one gene at a time. A gene whose within-clade distances are not smaller than
its between-clade distances adds noise (saturated, badly aligned or paralogous) and is a candidate to drop. Genes
seen in fewer than `MIN_PAIRS` sampled within-clade pairs are left out because their value is noisy.

**How to read it.** A few genes slightly below the rest is normal. A clear low tail (several genes under ~0.6)
suggests removing them and re-running. A uniformly low distribution points at the labels or the alignment step.
The heatmap shows the genes with the lowest mean separation at order level, for every variant.

In [ ]:
gene_separation = pl.concat([DATA[l]["gene_separation"] for l in LABELS])
gene_separation.write_csv(OUT / "gene_separation.tsv", separator="\t")
usable = gene_separation.filter((pl.col("within_pairs_with_data") >= MIN_PAIRS) & pl.col("separation").is_not_nan())

fig, axes = plt.subplots(1, len(RANKS), figsize=(3.2 * len(RANKS), 3.2), sharey=True)
for axis, rank in zip(axes, RANKS):
    for label in LABELS:
        values = usable.filter((pl.col("rank") == rank) & (pl.col("variant") == label))["separation"]
        axis.hist(values, bins=25, range=(0.5, 1.0), histtype="step", color=COLOURS[label], label=label)
    axis.set_title(rank)
    axis.set_xlabel("gene separation")
axes[0].set_ylabel("genes")
axes[0].legend(fontsize=7)
finish(fig, "gene_separation_distribution")

In [ ]:
N_GENES = 25
order = usable.filter(pl.col("rank") == "order")
worst_genes = (order.group_by("gene").agg(pl.col("separation").mean().alias("mean"))
               .sort("mean").head(N_GENES)["gene"].to_list())
grid = np.full((len(worst_genes), len(LABELS)), np.nan)
for j, label in enumerate(LABELS):
    lookup = dict(zip(*order.filter(pl.col("variant") == label).select("gene", "separation").to_dict(as_series=False).values()))
    grid[:, j] = [lookup.get(g, np.nan) for g in worst_genes]

fig, axis = plt.subplots(figsize=(1.2 * len(LABELS) + 3, 0.3 * N_GENES + 1.5))
image = axis.imshow(grid, aspect="auto", vmin=0.5, vmax=1.0, cmap="viridis")
axis.set_xticks(range(len(LABELS)))
axis.set_xticklabels(LABELS, rotation=45, ha="right")
axis.set_yticks(range(len(worst_genes)))
axis.set_yticklabels(worst_genes, fontsize=8)
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        if not np.isnan(grid[i, j]):
            axis.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white")
plt.colorbar(image, label="separation")
axis.set_title(f"{N_GENES} lowest-separation genes (order level)")
finish(fig, "gene_separation_worst_genes")

## 5. Single vs multi copy

For each `derep` value the same genomes are used in the `single` and `multi` alignments, so each gene can be
compared with itself: every point is one gene, its separation in the `single` alignment on the x axis and in the
`multi` alignment on the y axis. Points above the diagonal are genes the extra Duplicated copies **improve**.

**How to read it.** A cloud on the diagonal means the extra copies change little. A cloud clearly above it says the
extra copies add real signal; clearly below says the top-score rule is adding paralogs. Differences in the third
decimal are noise.

In [ ]:
PAIRED = [d for d in dict.fromkeys(v["derep"] for v in VARIANTS)
          if f"{d}/single" in DATA and f"{d}/multi" in DATA]
COMPARED = ["order", "family"]

if not PAIRED:
    print("No derep value has both a single and a multi alignment.")
else:
    fig, axes = plt.subplots(len(PAIRED), len(COMPARED), figsize=(4.5 * len(COMPARED), 4.2 * len(PAIRED)), squeeze=False)
    for row, derep in enumerate(PAIRED):
        for col, rank in enumerate(COMPARED):
            pick = lambda c: (usable.filter((pl.col("variant") == f"{derep}/{c}") & (pl.col("rank") == rank))
                              .select("gene", pl.col("separation").alias(c)))
            both = pick("single").join(pick("multi"), on="gene")
            axis = axes[row, col]
            axis.scatter(both["single"], both["multi"], s=10, alpha=0.7)
            axis.plot([0.5, 1], [0.5, 1], color="grey", linestyle=":")
            better = float((both["multi"] > both["single"]).mean()) if both.height else float("nan")
            axis.set_title(f"{derep} / {rank}: multi better for {better:.0%} of {both.height} genes", fontsize=9)
            axis.set_xlabel("separation, single")
            axis.set_ylabel("separation, multi")
    finish(fig, "single_vs_multi")

## 6. Multicopy genes: is the top-scoring copy a good choice?

When a BUSCO gene is *Duplicated* the aggregation keeps only the copy with the highest BUSCO score. That is a
heuristic: the best-scoring copy is not necessarily the orthologous one (a paralog can score higher).

Test: for randomly chosen Duplicated genes, compare each copy with **the same gene in related genomes** (same
family, else order, class, phylum; up to 25 peers) using 3-mer similarity as a cheap proxy for orthology, and ask
whether the chosen (top-score) copy is also the copy most similar to its peers. The peers come from each `multi`
variant's merged gene files.

* **agree** - share of genes where both choices are the same copy. High = the heuristic is fine.
* **similarity gap** - when they disagree, how much more peer-like the better copy is (3-mer Jaccard). A large
  gap means the heuristic picks a clearly worse copy.
* **score ratio** - runner-up score / top score. Near 1 means the two copies are nearly tied, so the score carries
  little information.

If *agree* is much below ~90% **and** the gaps are large, replace the rule with "choose the copy closest to the
clade" (or build the gene tree and pick the orthologous copy).

In [ ]:
# BUSCO full tables of every genome: (accession, path)
tables = []
for entry in PARAMS["busco_manifests"]:
    manifest, base = Path(entry["manifest"]), Path(entry["base"])
    if not manifest.exists():
        print("missing manifest:", manifest)
        continue
    qc = pl.read_csv(manifest, separator="\t")
    tables += [(accession, str(Path(result) if Path(result).is_absolute() else base / result))
               for accession, result in zip(qc["genome_acc"], qc["busco_result"])]

multi_variants = [v for v in VARIANTS if v["copies"] == "multi"] or VARIANTS[:1]
case_rows = []
for variant in multi_variants:
    cases = lib.multicopy_analysis(tables, taxonomy, variant["merged_dir"],
                                   max_cases=PARAMS["max_multicopy_cases"], seed=2) if tables else []
    case_rows += [{"variant": variant["label"],
                   **{k: case[k] for k in ("genome", "gene", "n_copies", "agree", "similarity_gap",
                                           "score_ratio", "copy_identity", "peer_rank", "n_peers")}}
                  for case in cases]
multicopy = pl.DataFrame(case_rows)
print(f"{multicopy.height} Duplicated genes evaluated in {len(multi_variants)} variant(s)")
if multicopy.height:
    multicopy.write_csv(OUT / "multicopy_cases.tsv", separator="\t")

In [ ]:
if multicopy.height:
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
    agree = multicopy.group_by("variant", maintain_order=True).agg(pl.col("agree").mean())
    axes[0].bar(agree["variant"], agree["agree"], color=[COLOURS[v] for v in agree["variant"]])
    axes[0].set_ylim(0, 1)
    axes[0].set_title("top-score copy = most peer-like copy")
    axes[0].set_ylabel("share of genes")
    axes[1].hist(multicopy["similarity_gap"], bins=30)
    axes[1].set_title("peer-similarity gap\n(best peer-like copy - chosen copy)")
    axes[2].scatter(multicopy["score_ratio"], multicopy["similarity_gap"], s=8, alpha=0.6,
                    c=np.where(multicopy["agree"], "tab:blue", "tab:red"))
    axes[2].set_xlabel("score ratio (runner-up / top)")
    axes[2].set_ylabel("gap (red = copies disagree)")
    axes[3].hist(multicopy["copy_identity"], bins=30)
    axes[3].set_title("similarity between the two best copies")
    finish(fig, "multicopy_check")
else:
    print("No multicopy cases (is use_multi_copy enabled, and do genomes have Duplicated genes?)")

## 7. Summary

One row per variant. Tables and PNG copies of the figures are written next to the executed notebook
(`tables/tree_diagnostics/`).

In [ ]:
summary = pl.DataFrame([{
    "variant": label,
    "genomes": DATA[label]["occupancy"].height,
    "median_genes": DATA[label]["occupancy"]["genes"].median(),
    "rows_90pct_gaps": int((DATA[label]["occupancy"]["gap_fraction"] >= 0.9).sum()),
    "separation_order": round(DATA[label]["report"]["order"]["summary"]["separation_pdist"], 3),
    "separation_genus": round(DATA[label]["report"]["genus"]["summary"]["separation_pdist"], 3),
    "genes_below_0.6_order": int(usable.filter((pl.col("variant") == label) & (pl.col("rank") == "order")
                                                & (pl.col("separation") < 0.6)).height),
} for label in LABELS])
show_all(summary)
if multicopy.height:
    print(f"Multicopy: top-score copy is the most peer-like in {multicopy['agree'].mean():.0%} of {multicopy.height} genes")